# VentriQ — Phase 2: ECG Signal Preprocessing & Quality Control

**Objective:** Implement and validate a safe, reproducible signal preprocessing and quality control module for 12-lead ECGs.

### Key Tasks:
1. Numerical sanity checking (dimensions, NaNs, infinities, empty arrays).
2. Lead quality fault evaluation (flatline leads, rail clipping, powerline hum, SNR estimation).
3. Configurable zero-phase forward-backward Butterworth filtering ($0.5 - 40\text{ Hz}$) with $0\text{ ms}$ delay.
4. Morphology-preserving normalization experiments (lead z-score, global z-score, min-max, robust IQR).
5. Keeping distinct: (1) Original Released, (2) Quality-Checked, (3) Filtered, and (4) Normalized representations.

In [ ]:
from pathlib import Path
import sys

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
import yaml

from src.data.echonext import (
    EchoNextDataset,
    create_mock_echonext_dataset,
    STANDARD_12_LEADS,
    SAMPLING_RATE_HZ,
    EXPECTED_TIME_STEPS,
    RECORDING_DURATION_SEC,
)
from src.preprocessing.quality import ECGQualityEvaluator, QualityReport
from src.preprocessing.signal import ECGSignalPreprocessor, ProcessedECG
from src.utils.paths import get_raw_echonext_dir, CONFIGS_DIR

print("Python ready. NumPy:", np.__version__)

## 1. Load Dataset Record and Inspect Raw Waveform

In [ ]:
data_dir = get_raw_echonext_dir()
if not (data_dir / "EchoNext_metadata_100k.csv").exists() and not (data_dir / "echonext_metadata_100k.csv").exists():
    create_mock_echonext_dataset(data_dir, num_train=70, num_val=15, num_test=15, seed=42)

dataset = EchoNextDataset(data_dir)
raw_ecg = dataset.get_waveform(0)
lvef = dataset.get_lvef(0)

print(f"Loaded ECG record 0 | LVEF: {lvef:.1f}%")
print(f"Raw array shape: {raw_ecg.shape} (Time steps x Leads)")
print(f"Dtype: {raw_ecg.dtype} | Range: [{raw_ecg.min():.2f}, {raw_ecg.max():.2f}]")

## 2. Quality Evaluation on Clean Signal

In [ ]:
evaluator = ECGQualityEvaluator()
report = evaluator.evaluate(raw_ecg)

print(f"Is Valid: {report.is_valid}")
print(f"Is Usable: {report.is_usable}")
print(f"Overall SNR: {report.overall_snr_db:.2f} dB")
print(f"Flatline Leads: {report.flatline_leads}")
print(f"Clipped Leads: {report.clipped_leads}")
print(f"Low SNR Leads: {report.noisy_leads}")

## 3. Lead Fault Simulation & Detection
We inject controlled anomalies to verify detection:
- **Lead V1 (index 6):** Disconnection / Flatline
- **Lead aVR (index 3):** Rail saturation / Clipping
- **Lead III (index 2):** High-frequency EMG noise

In [ ]:
faulty_ecg = raw_ecg.copy()
# 1. Flatline Lead V1
faulty_ecg[:, 6] = 0.0
# 2. Clip Lead aVR at maximum
max_val = np.max(faulty_ecg[:, 3])
faulty_ecg[200:230, 3] = max_val
# 3. Noise on Lead III
faulty_ecg[:, 2] += np.random.normal(0, 0.8, EXPECTED_TIME_STEPS).astype(np.float32)

fault_report = evaluator.evaluate(faulty_ecg)
print("Detected Flatline Leads:", fault_report.flatline_leads)
print("Detected Clipped Leads: ", fault_report.clipped_leads)
print("Detected Noisy Leads:   ", fault_report.noisy_leads)
print(f"Usable under max 2 faulty leads rule: {fault_report.is_usable}")

## 4. Zero-Phase Filtering & Delay Verification
Forward-backward filtering (`sosfiltfilt`) guarantees zero phase distortion ($0\text{ ms}$ delay).

In [ ]:
preprocessor = ECGSignalPreprocessor(
    apply_filter=True,
    lowcut_hz=0.5,
    highcut_hz=40.0,
    filter_order=4,
    apply_notch=True,
    notch_frequencies_hz=[50.0, 60.0],
)

processed = preprocessor.process(raw_ecg)

# Plot zoomed 2.5-second Lead II comparison to verify peak alignment
t_zoom = np.linspace(0, 2.5, int(2.5 * SAMPLING_RATE_HZ), endpoint=False)
n_zoom = len(t_zoom)

plt.figure(figsize=(10, 4), dpi=120)
plt.plot(t_zoom, processed.original[:n_zoom, 1], label="Original (Lead II)", color="#2980b9", lw=1.5)
plt.plot(t_zoom, processed.filtered[:n_zoom, 1], label="Zero-Phase Filtered", color="#c0392b", linestyle="--", lw=1.5)
plt.title("Lead II QRS Peak Alignment: Zero Phase Shift (0 ms Delay)", fontweight="bold")
plt.xlabel("Time (seconds)")
plt.ylabel("Normalized Amplitude")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Spectral Analysis: Power Spectral Density (PSD)

In [ ]:
lead_idx = 1  # Lead II
f_orig, psd_orig = scipy_signal.welch(processed.original[:, lead_idx], fs=SAMPLING_RATE_HZ, nperseg=500)
f_filt, psd_filt = scipy_signal.welch(processed.filtered[:, lead_idx], fs=SAMPLING_RATE_HZ, nperseg=500)

plt.figure(figsize=(9, 4), dpi=120)
plt.semilogy(f_orig, psd_orig, label="Original PSD", color="#2980b9")
plt.semilogy(f_filt, psd_filt, label="Filtered PSD (0.5 - 40 Hz)", color="#e74c3c", linestyle="--")
plt.axvline(0.5, color="gray", linestyle=":", label="Lowcut (0.5 Hz)")
plt.axvline(40.0, color="gray", linestyle=":", label="Highcut (40.0 Hz)")
plt.title("Power Spectral Density (PSD) Comparison — Lead II", fontweight="bold")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density (V²/Hz)")
plt.grid(True, linestyle=":", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Normalization Methods Comparison

In [ ]:
norm_zscore = preprocessor.normalize(raw_ecg, method="lead_zscore")
norm_minmax = preprocessor.normalize(raw_ecg, method="minmax")
norm_robust = preprocessor.normalize(raw_ecg, method="robust")

plt.figure(figsize=(11, 4), dpi=120)
plt.plot(t_zoom, norm_zscore[:n_zoom, 1], label="Lead Z-Score (μ=0, σ=1)", color="#27ae60")
plt.plot(t_zoom, norm_minmax[:n_zoom, 1], label="Min-Max ([-1, 1])", color="#8e44ad", linestyle=":")
plt.plot(t_zoom, norm_robust[:n_zoom, 1], label="Robust IQR", color="#d35400", linestyle="--")
plt.title("Comparison of Morphology-Preserving Normalization Options", fontweight="bold")
plt.xlabel("Time (seconds)")
plt.ylabel("Scaled Amplitude")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Phase 2 Verification Signoff

1. **Signal Validation:** Numerical sanity checking handles NaNs, infinities, empty arrays, and dimensions.
2. **Lead Fault Detection:** Accurately flags flatline leads, rail clipping, and high noise floor without crashing.
3. **Zero-Phase Filtering:** Butterworth SOS forward-backward filtering removes baseline wander and high-frequency noise with zero peak delay.
4. **Multi-Stage Integrity:** Original released waveforms are kept distinct and unmodified by default.